In [4]:
from transformers import BertTokenizer, BertForQuestionAnswering
import torch
import warnings
warnings.filterwarnings("ignore")

In [3]:
model_name="bert-base-uncased"

In [5]:
tokenizer=BertTokenizer.from_pretrained(model_name)
model=BertForQuestionAnswering.from_pretrained(model_name)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 5323.68it/s]
BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expec

In [11]:
def predict_answer(context, question):
    """
    context=metin
    question=soru
    Amac: metin içeriklerinde soruyu bulma
    """
    encoding = tokenizer(question, context, return_tensors="pt", max_length=512, truncation=True)
    
    # Properly extract the required tensors
    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    
    # Safely get token_type_ids (Some models like DistilBERT don't return them)
    token_type_ids = encoding.get("token_type_ids")

    with torch.no_grad():
        # Pass all available tensors to the model
        if token_type_ids is not None:
            start_scores, end_scores = model(
                input_ids, 
                attention_mask=attention_mask, 
                token_type_ids=token_type_ids, 
                return_dict=False
            )[:2] # Slice to ensure we only grab the first two outputs (logits)
        else:
            start_scores, end_scores = model(
                input_ids, 
                attention_mask=attention_mask, 
                return_dict=False
            )[:2]

    # Get the most likely start and end positions
    start_index = torch.argmax(start_scores, dim=1).item()
    end_index = torch.argmax(end_scores, dim=1).item()

    # Edge case: If the start index is after the end index, the model failed to find a valid answer
    if start_index > end_index:
        return "No valid answer found."

    # Convert the token IDs back to a readable string
    answer_tokens = tokenizer.convert_ids_to_tokens(input_ids[0][start_index:end_index+1])
    answer = tokenizer.convert_tokens_to_string(answer_tokens)
    
    return answer

# Test the function
question = "What is the capital of France?"
context = "Paris is the capital of France. It is known for its art, culture, and history."
answer = predict_answer(context, question)

print(f"Question: {question}")
print(f"Answer: {answer}")



Question: What is the capital of France?
Answer: the capital of france?
